In [ ]:
import os, time, warnings
import numpy as np
import pandas as pd
from scipy.special import logsumexp
warnings.filterwarnings("ignore")


SAMPLED_ROOT = r"D:\doc\2026\FINAL_CODE\data\sampled_small"
CSV_PATH = (
    r"D:/doc/2026/FINAL_CODE/src/codeStat/preModelling/"
    r"samplingStratified/out_samplingStratified/dataSampledSmall.csv"
)
DIR_SAVE = (
    r"D:/doc/2026/FINAL_CODE/src/codeStat/modelling/"
    r"voiceModelling/srcVoiceModelling/GMM_TDNN/out/test_5"
)
os.makedirs(DIR_SAVE, exist_ok=True)

OUT_WER_CSV     = os.path.join(DIR_SAVE, "HasilWER_TDNN_v5b.csv")
OUT_DEC_CSV     = os.path.join(DIR_SAVE, "HasilDekode_TDNN_v5b.csv")
OUT_HISTORY_CSV = os.path.join(DIR_SAVE, "HasilTraining_TDNN_v5b.csv")
MODEL_PATH      = os.path.join(DIR_SAVE, "tdnn_v5b_best.npy")
NORM_PATH       = os.path.join(DIR_SAVE, "mfcc_global_norm.npy")

COL_PATH_DATA = "Path_Data"
COL_SENTENCES = "sentences"

SR=16000; N_MFCC=13; N_FFT=512; HOP_LENGTH=160
WIN_LENGTH=400; N_MELS=26; PRE_EMPH=0.97
FMIN=0.0; FMAX=4000.0; DELTA_WIDTH=2
INPUT_DIM = N_MFCC * 3   # 39

TDNN_CONTEXTS = [[-2,2],[-2,0],[-3,3]]
HIDDEN_DIMS   = [256, 256, 256]
DROPOUT_RATE  = 0.2
LR_INIT       = 1e-3
LR_WARMUP     = 1e-5
GRAD_CLIP     = 1.0

BATCH_SIZE   = 16
MAX_EPOCHS   = 60
EARLY_STOP   = 12
WARMUP_EPOCH = 5
LR_DECAY     = 0.5
LR_PATIENCE  = 5
LR_MIN       = 1e-6

TRAIN_RATIO = 0.80
VAL_RATIO   = 0.10
RANDOM_SEED = 42
BLANK       = 0
BEAM_WIDTH  = 10



_ANGKA_JAWA = {
    0:'nol',1:'siji',2:'loro',3:'telu',4:'papat',5:'lima',
    6:'enem',7:'pitu',8:'wolu',9:'songo',10:'sepuluh',
    11:'sewelas',12:'rolas',13:'telulas',14:'patbelas',
    15:'limolas',16:'nembelas',17:'pitulas',18:'wolulas',
    19:'sangalas',20:'rongpuluh',
}

def _angka_ke_jawa(n: int) -> str:
    if n in _ANGKA_JAWA: return _ANGKA_JAWA[n]
    if n < 100:
        p = _ANGKA_JAWA.get(n//10, str(n//10)) + "puluh"
        s = n % 10
        return p if s==0 else p + _ANGKA_JAWA.get(s, str(s))
    if n < 1000:
        r = n // 100; sisa = n % 100
        pfx = "satus" if r==1 else _ANGKA_JAWA.get(r, str(r)) + "atus"
        return pfx if sisa==0 else pfx + _angka_ke_jawa(sisa)
    r = n // 1000; sisa = n % 1000
    pfx = "sewu" if r==1 else _ANGKA_JAWA.get(r, str(r)) + "ewu"
    return pfx if sisa==0 else pfx + _angka_ke_jawa(sisa)

def normalize_text(text: str) -> str:
    import re
    text = text.lower().strip()
    text = re.sub(r'\d+', lambda m: ' ' + _angka_ke_jawa(int(m.group())) + ' ', text)
    text = re.sub(r"[^a-zéèêâ\s]", ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

DIGRAPHS = {'ng': 'ŋ', 'ny': 'ɲ', 'th': 'tʰ', 'dh': 'dʰ'}

def javanese_g2p(text: str) -> list:
    text = normalize_text(text)
    phonemes = []; i = 0
    while i < len(text):
        ch = text[i]
        if ch == ' ':
            if phonemes and phonemes[-1] != '<sp>': phonemes.append('<sp>')
            i += 1; continue
        if not ch.isalpha() and ch not in 'éèâê': i += 1; continue
        if i+1 < len(text):
            two = text[i:i+2]
            if two in DIGRAPHS: phonemes.append(DIGRAPHS[two]); i+=2; continue
        phonemes.append(ch); i += 1
    return phonemes

def build_phoneme_vocab(sentences):
    all_phn = set()
    for s in sentences: all_phn.update(javanese_g2p(s))
    vocab  = ['<blank>'] + sorted(all_phn)
    phn2id = {p: i for i,p in enumerate(vocab)}
    id2phn = {i: p for p,i in phn2id.items()}
    return vocab, phn2id, id2phn

def encode_phoneme(sentence, phn2id):
    return [phn2id[p] for p in javanese_g2p(sentence) if p in phn2id]

def phonemes_to_text(phonemes):
    r = []
    for p in phonemes:
        if p=='<sp>': r.append(' ')
        elif p=='ŋ':  r.append('ng')
        elif p=='ɲ':  r.append('ny')
        elif p=='tʰ': r.append('th')
        elif p=='dʰ': r.append('dh')
        else:         r.append(p)
    return ''.join(r).strip()


# ─────────────────────────────────────────────────────────────
#  EKSTRAKSI MFCC (mentah, tanpa normalisasi per-file)
# ─────────────────────────────────────────────────────────────

try:
    import librosa; LIBROSA_OK = True
except ImportError:
    LIBROSA_OK = False
try:
    from scipy.io import wavfile; SCIPY_OK = True
except ImportError:
    SCIPY_OK = False

def extract_features_raw(wav_path):
    if not os.path.isfile(wav_path): raise FileNotFoundError(wav_path)
    return _librosa_raw(wav_path) if LIBROSA_OK else _manual_raw(wav_path)

def _librosa_raw(wav_path):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    if len(y)==0: raise ValueError("Audio kosong")
    y    = np.append(y[0], y[1:] - PRE_EMPH*y[:-1])
    mfcc = librosa.feature.mfcc(y=y, sr=SR, n_mfcc=N_MFCC, n_fft=N_FFT,
               hop_length=HOP_LENGTH, win_length=WIN_LENGTH,
               n_mels=N_MELS, fmin=FMIN, fmax=FMAX, window="hamming")
    d1 = librosa.feature.delta(mfcc, width=DELTA_WIDTH*2+1)
    d2 = librosa.feature.delta(mfcc, order=2, width=DELTA_WIDTH*2+1)
    return np.vstack([mfcc,d1,d2]).T.astype(np.float32)

def _delta_vec(feat, width=2):
    T=feat.shape[0]; denom=2.0*sum(n*n for n in range(1,width+1))
    delta=np.zeros_like(feat)
    for n in range(1,width+1):
        delta += n*(feat[np.clip(np.arange(T)+n,0,T-1)] -
                    feat[np.clip(np.arange(T)-n,0,T-1)])
    return delta/denom

def _manual_raw(wav_path):
    if not SCIPY_OK: raise ImportError("Butuh librosa atau scipy")
    sr, audio = wavfile.read(wav_path)
    if audio.dtype==np.int16: audio=audio/32768.0
    elif audio.dtype==np.int32: audio=audio/2147483648.0
    audio=audio.astype(np.float32)
    if sr!=SR:
        n=int(len(audio)*SR/sr)
        audio=np.interp(np.linspace(0,len(audio)-1,n),np.arange(len(audio)),audio)
    y=np.append(audio[0],audio[1:]-PRE_EMPH*audio[:-1])
    T_fr=1+(len(y)-WIN_LENGTH)//HOP_LENGTH
    idx=np.arange(WIN_LENGTH)[None,:]+np.arange(T_fr)[:,None]*HOP_LENGTH
    frames=y[idx]*np.hamming(WIN_LENGTH)
    fft_m=np.abs(np.fft.rfft(frames,n=N_FFT))
    def hz2mel(f): return 1127*np.log1p(f/700)
    def mel2hz(m): return 700*(np.exp(m/1127)-1)
    pts=mel2hz(np.linspace(hz2mel(FMIN),hz2mel(FMAX),N_MELS+2))
    bins=np.floor((N_FFT+1)*pts/SR).astype(int)
    fb=np.zeros((N_MELS,N_FFT//2+1))
    for m in range(1,N_MELS+1):
        for k in range(bins[m-1],bins[m]):
            fb[m-1,k]=(k-bins[m-1])/(bins[m]-bins[m-1]+1e-10)
        for k in range(bins[m],bins[m+1]):
            fb[m-1,k]=(bins[m+1]-k)/(bins[m+1]-bins[m]+1e-10)
    log_mel=np.log10(np.maximum(fft_m**2@fb.T,1e-10))
    N_=N_MELS; ni=np.arange(N_MFCC)[:,None]; ki=np.arange(N_)[None,:]+0.5
    mfcc=log_mel@(np.cos(np.pi*ni*ki/N_)*np.sqrt(2/N_)).T
    d1=_delta_vec(mfcc,DELTA_WIDTH); d2=_delta_vec(d1,DELTA_WIDTH)
    return np.concatenate([mfcc,d1,d2],axis=1).astype(np.float32)


# ─────────────────────────────────────────────────────────────
#  [FIX 2] NORMALISASI MFCC GLOBAL
# ─────────────────────────────────────────────────────────────

def compute_global_norm(sequences, train_ids):
    print("  Menghitung normalisasi MFCC global dari train set ...")
    all_frames = np.concatenate([sequences[aid] for aid in train_ids], axis=0)
    mean = all_frames.mean(axis=0).astype(np.float32)
    std  = all_frames.std(axis=0).astype(np.float32) + 1e-8
    print(f"  mean [{mean.min():.2f}, {mean.max():.2f}]  "
          f"std [{std.min():.4f}, {std.max():.2f}]")
    return mean, std

def apply_global_norm(sequences, mean, std):
    return {aid: (feat - mean) / std for aid, feat in sequences.items()}


# ─────────────────────────────────────────────────────────────
#  DATA LOADER
# ─────────────────────────────────────────────────────────────

def find_wav(audio_id):
    for root,_,files in os.walk(SAMPLED_ROOT):
        for f in files:
            if os.path.splitext(f)[0]==audio_id and f.lower().endswith(".wav"):
                return os.path.join(root,f)
    return ""

def load_data(csv_path):
    print(f"  Membaca {csv_path} ...")
    df = pd.read_csv(csv_path)
    print(f"  Shape CSV : {df.shape}")
    print(f"  Kolom     : {list(df.columns)}")
    for col in [COL_PATH_DATA, COL_SENTENCES]:
        if col not in df.columns: raise ValueError(f"Kolom '{col}' tidak ada.")

    sentences_norm = [normalize_text(str(s)) for s in df[COL_SENTENCES].tolist()]
    vocab, phn2id, id2phn = build_phoneme_vocab(sentences_norm)
    print(f"  Vocab fonem : {len(vocab)-1}  (+1 blank → total {len(vocab)})")
    print(f"  Fonem       : {sorted(set(vocab)-{'<blank>'})}")

    sequences_raw, targets, word_refs = {}, {}, {}
    skipped = 0

    print(f"  Mengekstrak MFCC dari {len(df)} file .wav ...")
    for idx, row in df.iterrows():
        audio_id = os.path.splitext(os.path.basename(str(row[COL_PATH_DATA])))[0]
        kalimat  = normalize_text(str(row[COL_SENTENCES]))
        words    = kalimat.split()
        wav_path = str(row[COL_PATH_DATA])
        if not os.path.isfile(wav_path): wav_path = find_wav(audio_id)
        if not wav_path: skipped+=1; continue
        try:
            feats  = extract_features_raw(wav_path)
            T      = feats.shape[0]
            target = encode_phoneme(kalimat, phn2id)
            if T==0 or not target: skipped+=1; continue
            if T < 2*len(target)+1: skipped+=1; continue
            sequences_raw[audio_id] = feats
            targets[audio_id]       = target
            word_refs[audio_id]     = words
        except Exception as e:
            print(f"  ⚠ {audio_id}: {e}"); skipped+=1; continue
        if (idx+1)%200==0 or (idx+1)==len(df):
            print(f"    [{idx+1}/{len(df)}] diproses, {skipped} dilewati")

    actual_dim = next(iter(sequences_raw.values())).shape[1]
    lens = [len(targets[a]) for a in sequences_raw]
    print(f"\n  Audio berhasil : {len(sequences_raw)} | dilewati : {skipped}")
    print(f"  Total frame    : {sum(v.shape[0] for v in sequences_raw.values()):,}")
    print(f"  Input dim      : {actual_dim} | Output dim : {len(vocab)}")
    print(f"  Panjang fonem  : min={min(lens)}, mean={np.mean(lens):.1f}, max={max(lens)}")
    return sequences_raw, targets, word_refs, phn2id, id2phn, vocab, actual_dim


# ─────────────────────────────────────────────────────────────
#  CTC LOSS + GRADIENT (tidak berubah)
# ─────────────────────────────────────────────────────────────

def _softmax(logits):
    x = logits - logits.max(axis=1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=1, keepdims=True)

def ctc_loss_and_grad(logits, targets):
    T, V = logits.shape; L = len(targets); S = 2*L+1; NEG = -1e9
    l_ = np.zeros(S, dtype=np.int32)
    for i,c in enumerate(targets): l_[2*i+1] = c
    probs     = _softmax(logits).astype(np.float64)
    log_probs = np.log(probs+1e-30); lp_ext = log_probs[:,l_]
    can_skip = np.zeros(S,dtype=bool)
    for s in range(2,S):
        if l_[s]!=BLANK and l_[s]!=l_[s-2]: can_skip[s]=True
    can_skip_bk = np.zeros(S,dtype=bool)
    for s in range(S-2):
        if l_[s+2]!=BLANK and l_[s+2]!=l_[s]: can_skip_bk[s]=True
    alpha = np.full((T,S),NEG); alpha[0,0]=lp_ext[0,0]
    if S>1: alpha[0,1]=lp_ext[0,1]
    for t in range(1,T):
        s0=alpha[t-1]; s1=np.empty(S); s1[0]=NEG; s1[1:]=alpha[t-1,:-1]
        s2=np.empty(S); s2[:2]=NEG; s2[2:]=alpha[t-1,:-2]
        alpha[t]=np.where(can_skip,np.logaddexp(np.logaddexp(s0,s1),s2),
                          np.logaddexp(s0,s1))+lp_ext[t]
    beta = np.full((T,S),NEG); beta[T-1,S-1]=0.0
    if S>1: beta[T-1,S-2]=0.0
    for t in range(T-2,-1,-1):
        lp=lp_ext[t+1]; s0=beta[t+1]+lp
        s1=np.empty(S); s1[-1]=NEG; s1[:-1]=beta[t+1,1:]+lp[1:]
        s2=np.empty(S); s2[-2:]=NEG; s2[:-2]=beta[t+1,2:]+lp[2:]
        beta[t]=np.where(can_skip_bk,np.logaddexp(np.logaddexp(s0,s1),s2),
                         np.logaddexp(s0,s1))
    ll_terms=[alpha[T-1,S-1]]
    if S>1: ll_terms.append(alpha[T-1,S-2])
    ll = float(logsumexp(ll_terms))
    if not np.isfinite(ll) or ll < -1e7:
        return 0.0, np.zeros((T,V),dtype=np.float32)
    loss = float(-ll/T)
    ab=alpha+beta; log_numer=np.full((T,V),NEG)
    for s in range(S):
        log_numer[:,l_[s]]=np.logaddexp(log_numer[:,l_[s]],ab[:,s])
    term = np.exp(np.clip(log_numer-ll,-500,0))
    return loss, (probs-term).astype(np.float32)

────────────────────────────────────────────────────────

class LayerNorm:
    """
    Layer Normalization sederhana untuk sequence (T, D).
    Normalize sepanjang dimensi D (feature) untuk setiap timestep.
    """
    def __init__(self, dim, eps=1e-5):
        self.eps   = eps
        self.gamma = np.ones(dim,  dtype=np.float32)
        self.beta  = np.zeros(dim, dtype=np.float32)
        # Adam state
        self.mg=np.zeros(dim,dtype=np.float32); self.vg=np.zeros(dim,dtype=np.float32)
        self.mb=np.zeros(dim,dtype=np.float32); self.vb=np.zeros(dim,dtype=np.float32)
        self.t=0
        # cache untuk backward
        self._xhat=None; self._std=None
        self._dg_acc=np.zeros(dim,dtype=np.float32)
        self._db_acc=np.zeros(dim,dtype=np.float32)
        self._n_acc=0

    def forward(self, x):
        # x: (T, D)
        mean = x.mean(axis=1, keepdims=True)
        var  = x.var(axis=1,  keepdims=True)
        std  = np.sqrt(var + self.eps)
        xhat = (x - mean) / std
        self._xhat = xhat; self._std = std
        return self.gamma * xhat + self.beta

    def backward(self, dout, accumulate=True):
        # dout: (T, D)
        T, D = dout.shape
        dg  = (dout * self._xhat).sum(axis=0)
        db  = dout.sum(axis=0)
        dxhat = dout * self.gamma
        # backward normalisasi (identik BN tapi axis=1)
        dx = (1.0/D) * (1.0/self._std) * (
            D*dxhat
            - dxhat.sum(axis=1, keepdims=True)
            - self._xhat * (dxhat * self._xhat).sum(axis=1, keepdims=True)
        )
        if accumulate:
            self._dg_acc += dg; self._db_acc += db; self._n_acc += 1
        else:
            self._adam(dg, db)
        return dx

    def flush(self):
        if self._n_acc==0: return
        self._adam(self._dg_acc/self._n_acc, self._db_acc/self._n_acc)
        self._dg_acc[:]=0; self._db_acc[:]=0; self._n_acc=0

    def set_lr(self, lr): self.lr = lr

    def _adam(self, dg, db, lr=None, b1=0.9, b2=0.999, eps=1e-8):
        if lr is None: lr = getattr(self, 'lr', 1e-3)
        self.t += 1
        self.mg=b1*self.mg+(1-b1)*dg; self.vg=b2*self.vg+(1-b2)*dg**2
        self.gamma -= lr*(self.mg/(1-b1**self.t))/(np.sqrt(self.vg/(1-b2**self.t))+eps)
        self.mb=b1*self.mb+(1-b1)*db; self.vb=b2*self.vb+(1-b2)*db**2
        self.beta  -= lr*(self.mb/(1-b1**self.t))/(np.sqrt(self.vb/(1-b2**self.t))+eps)


# ─────────────────────────────────────────────────────────────
#  TDNN LAYER dengan LayerNorm (FIX: bukan BatchNorm)
# ─────────────────────────────────────────────────────────────

class TDNNLayer:
    def __init__(self, in_dim, out_dim, context, lr, use_ln=True):
        self.context = context
        self.ctx_len = context[1]-context[0]+1
        self.lr      = lr
        self.use_ln  = use_ln   # False untuk layer output
        fan_in       = in_dim*self.ctx_len
        self.W = (np.random.randn(fan_in, out_dim) *
                  np.sqrt(2.0/fan_in)).astype(np.float32)
        self.b = np.zeros(out_dim, dtype=np.float32)
        if use_ln:
            self.ln = LayerNorm(out_dim)
            self.ln.lr = lr
        self.mW=np.zeros_like(self.W); self.vW=np.zeros_like(self.W)
        self.mb=np.zeros_like(self.b); self.vb=np.zeros_like(self.b)
        self.t=0; self._X=None
        self._dW_acc=np.zeros_like(self.W)
        self._db_acc=np.zeros_like(self.b)
        self._n_acc=0

    def splice(self, X):
        T=X.shape[0]; idx=np.arange(T)
        return np.concatenate(
            [X[np.clip(idx+off,0,T-1)]
             for off in range(self.context[0],self.context[1]+1)],axis=1)

    def forward(self, Xsp):
        self._X = Xsp
        z = Xsp@self.W + self.b
        if self.use_ln: z = self.ln.forward(z)
        return z

    def backward(self, dout, accumulate=True):
        T = len(dout)
        if self.use_ln: dout = self.ln.backward(dout, accumulate=accumulate)
        dW   = self._X.T@dout/T
        db   = dout.mean(0)
        dXsp = dout@self.W.T
        if accumulate:
            self._dW_acc+=dW; self._db_acc+=db; self._n_acc+=1
        else:
            self._adam(dW, db)
        B,D_tot=dXsp.shape; D=D_tot//self.ctx_len
        dX=np.zeros((B,D),dtype=np.float32)
        for k in range(self.ctx_len): dX+=dXsp[:,k*D:(k+1)*D]
        return dX

    def flush(self):
        if self._n_acc==0: return
        self._adam(self._dW_acc/self._n_acc, self._db_acc/self._n_acc)
        if self.use_ln: self.ln.flush()
        self._dW_acc[:]=0; self._db_acc[:]=0; self._n_acc=0

    def set_lr(self, lr):
        self.lr = lr
        if self.use_ln: self.ln.lr = lr

    def _adam(self, dW, db, b1=0.9, b2=0.999, eps=1e-8):
        self.t+=1
        self.mW=b1*self.mW+(1-b1)*dW; self.vW=b2*self.vW+(1-b2)*dW**2
        self.W-=self.lr*(self.mW/(1-b1**self.t))/(np.sqrt(self.vW/(1-b2**self.t))+eps)
        self.mb=b1*self.mb+(1-b1)*db; self.vb=b2*self.vb+(1-b2)*db**2
        self.b-=self.lr*(self.mb/(1-b1**self.t))/(np.sqrt(self.vb/(1-b2**self.t))+eps)


# ─────────────────────────────────────────────────────────────
#  MODEL TDNN
# ─────────────────────────────────────────────────────────────

class TDNNModel:
    def __init__(self, input_dim, hidden_dims, contexts, output_dim,
                 dropout=0.2, lr=1e-3):
        assert len(hidden_dims)==len(contexts)
        self.dropout=dropout; self.training=True
        self.layers=[]
        in_d=input_dim
        for h,ctx in zip(hidden_dims,contexts):
            # Hidden layers: pakai LayerNorm
            self.layers.append(TDNNLayer(in_d,h,ctx,lr,use_ln=True))
            in_d=h
        # [BUG A FIX] Layer output TIDAK pakai LayerNorm
        self.layers.append(TDNNLayer(in_d,output_dim,[0,0],lr,use_ln=False))
        self._rc=[]; self._dc=[]

    def set_training(self, mode):
        self.training=mode

    def set_lr(self, lr):
        for layer in self.layers: layer.set_lr(lr)

    def forward(self, X):
        self._rc=[]; self._dc=[]; h=X
        for layer in self.layers[:-1]:
            z=layer.forward(layer.splice(h))
            a=np.maximum(0.0,z)
            if self.training and self.dropout>0:
                mask=(np.random.rand(*a.shape)>self.dropout).astype(np.float32)
                a=a*mask/(1.0-self.dropout)
            else:
                mask=np.ones(a.shape,dtype=np.float32)
            self._rc.append(z); self._dc.append(mask); h=a
        out=self.layers[-1]
        return out.forward(out.splice(h))

    def ctc_step(self, X, target, accumulate=True):
        logits=self.forward(X)
        loss,dlogit=ctc_loss_and_grad(logits,target)
        if loss==0.0: return 0.0
        gnorm=np.sqrt((dlogit**2).sum())
        if gnorm>GRAD_CLIP: dlogit=dlogit*(GRAD_CLIP/gnorm)
        sc=1.0/(1.0-self.dropout+1e-9) if self.dropout>0 else 1.0
        dh=self.layers[-1].backward(dlogit,accumulate=accumulate)
        for i in reversed(range(len(self.layers)-1)):
            dh=dh*self._dc[i]*sc
            dh=dh*(self._rc[i]>0).astype(np.float32)
            dh=self.layers[i].backward(dh,accumulate=accumulate)
        return loss

    def flush(self):
        for layer in self.layers: layer.flush()

    def get_log_probs(self, X):
        self.set_training(False)
        p=_softmax(self.forward(X))
        return np.log(p+1e-30)

    def save(self, path):
        data={
            "w":[(l.W.copy(),l.b.copy()) for l in self.layers],
            "ln":[(l.ln.gamma.copy(),l.ln.beta.copy())
                  if l.use_ln else (None,None)
                  for l in self.layers],
        }
        np.save(path, data, allow_pickle=True)

    def load(self, path):
        d=np.load(path,allow_pickle=True).item()
        for i,(W,b) in enumerate(d["w"]):
            self.layers[i].W=W; self.layers[i].b=b
        if "ln" in d:
            for i,(g,bt) in enumerate(d["ln"]):
                if g is not None and self.layers[i].use_ln:
                    self.layers[i].ln.gamma=g
                    self.layers[i].ln.beta=bt
        print(f"  Model dimuat: {path}")


# ─────────────────────────────────────────────────────────────
#  SPLIT
# ─────────────────────────────────────────────────────────────

def split_dataset(ids):
    rng=np.random.default_rng(RANDOM_SEED)
    ids=list(ids); perm=rng.permutation(len(ids))
    n=len(ids); ntr=int(n*TRAIN_RATIO); nva=int(n*VAL_RATIO)
    tr=[ids[i] for i in perm[:ntr]]
    va=[ids[i] for i in perm[ntr:ntr+nva]]
    te=[ids[i] for i in perm[ntr+nva:]]
    print(f"  Split → train:{len(tr)}  val:{len(va)}  test:{len(te)}")
    return tr,va,te


# ─────────────────────────────────────────────────────────────
#  TRAINING
# ─────────────────────────────────────────────────────────────

def get_lr(epoch, no_improve):
    if epoch<=WARMUP_EPOCH:
        alpha=(epoch-1)/max(WARMUP_EPOCH-1,1)
        return LR_WARMUP+alpha*(LR_INIT-LR_WARMUP)
    return max(LR_INIT*(LR_DECAY**(no_improve//LR_PATIENCE)), LR_MIN)

def compute_wer_sample(model, ids, sequences, word_refs, id2phn,
                       n_sample=50, rng=None):
    if rng is None: rng=np.random.default_rng(42)
    sample=list(rng.choice(ids,min(n_sample,len(ids)),replace=False))
    model.set_training(False)
    tS=tD=tI=tN=0
    for aid in sample:
        lp=model.get_log_probs(sequences[aid])
        decoded=ctc_beam_decode(lp,id2phn,beam_width=5)
        hw=decoded.split(); rw=word_refs.get(aid,[])
        _,S,D,I,N=levenshtein(rw,hw)
        tS+=S;tD+=D;tI+=I;tN+=N
    return (tS+tD+tI)/max(tN,1)*100

def train(model, train_ids, val_ids, sequences, targets, word_refs, id2phn):
    rng=np.random.default_rng(RANDOM_SEED)
    hist=[]; best=float("inf"); pat=0; no_improve=0

    print(f"\n  Train: {len(train_ids)} | Val: {len(val_ids)} | "
          f"Epochs: {MAX_EPOCHS} | Batch: {BATCH_SIZE}")
    print(f"  LR: warmup {LR_WARMUP}→{LR_INIT} ({WARMUP_EPOCH}ep), "
          f"decay ×{LR_DECAY} tiap {LR_PATIENCE}ep")
    print(f"\n  {'Ep':>4}  {'LR':>8}  {'TrLoss':>8}  {'VaLoss':>8}  {'TrWER%':>7}  {'t':>7}")
    print(f"  {'-'*54}")

    for epoch in range(1,MAX_EPOCHS+1):
        t0=time.time()
        lr=get_lr(epoch,no_improve)
        model.set_lr(lr); model.set_training(True)

        perm=rng.permutation(len(train_ids))
        order=[train_ids[i] for i in perm]
        total_loss=0.0; n_batches=0

        for start in range(0,len(order),BATCH_SIZE):
            batch=order[start:start+BATCH_SIZE]
            batch_loss=0.0; valid=0
            for aid in batch:
                try:
                    lv=model.ctc_step(sequences[aid],targets[aid],accumulate=True)
                    if lv>0: batch_loss+=lv; valid+=1
                except Exception: continue
            model.flush()
            if valid>0: total_loss+=batch_loss/valid; n_batches+=1

        tr_loss=total_loss/max(n_batches,1)

        # [FIX 4] Val dengan training=False
        model.set_training(False)
        val_loss=0.0; val_n=0
        for aid in val_ids:
            try:
                logits=model.forward(sequences[aid])
                lv,_=ctc_loss_and_grad(logits,targets[aid])
                if lv>0: val_loss+=lv; val_n+=1
            except Exception: continue
        val_loss/=max(val_n,1)

        elapsed=time.time()-t0
        tr_wer_str="   -  "
        if epoch%5==0 or epoch==1:
            tr_wer=compute_wer_sample(model,train_ids,sequences,
                                      word_refs,id2phn,n_sample=50,rng=rng)
            tr_wer_str=f"{tr_wer:6.1f}%"

        hist.append({"epoch":epoch,"lr":round(lr,8),
                     "train_loss":round(tr_loss,6),
                     "val_loss":round(val_loss,6)})
        print(f"  {epoch:4d}  {lr:8.6f}  {tr_loss:8.4f}  "
              f"{val_loss:8.4f}  {tr_wer_str}  {elapsed:6.1f}s")

        if val_loss<best-1e-5:
            best=val_loss; pat=0; no_improve=0; model.save(MODEL_PATH)
        else:
            pat+=1; no_improve+=1
            if pat>=EARLY_STOP:
                print(f"\n  ✓ Early stop epoch {epoch} (best={best:.4f})")
                break
    return hist


# ─────────────────────────────────────────────────────────────
#  [FIX 6] BEAM SEARCH CTC DECODE
# ─────────────────────────────────────────────────────────────

def ctc_beam_decode(log_probs, id2phn, beam_width=10):
    T, V = log_probs.shape
    NEG  = -float('inf')
    beam = {(): (0.0, NEG)}   # seq → (log_Pb, log_Pnb)

    for t in range(T):
        new_beam = {}
        for seq,(pb,pnb) in beam.items():
            # Extend dengan blank
            lp_b=log_probs[t,BLANK]
            prev_pb,prev_pnb=new_beam.get(seq,(NEG,NEG))
            new_beam[seq]=(np.logaddexp(prev_pb,np.logaddexp(pb,pnb)+lp_b),
                           new_beam.get(seq,(NEG,NEG))[1])
            # Extend dengan non-blank
            for c in range(1,V):
                lp_c=log_probs[t,c]
                new_seq=seq+(c,)
                ep_nb=(pb+lp_c if seq and seq[-1]==c
                       else np.logaddexp(pb,pnb)+lp_c)
                prev_pb2,prev_pnb2=new_beam.get(new_seq,(NEG,NEG))
                new_beam[new_seq]=(prev_pb2,np.logaddexp(prev_pnb2,ep_nb))

        beam=dict(sorted(new_beam.items(),
                          key=lambda kv: np.logaddexp(kv[1][0],kv[1][1]),
                          reverse=True)[:beam_width])

    best_seq=max(beam.items(),
                 key=lambda kv: np.logaddexp(kv[1][0],kv[1][1]))[0]
    phonemes=[id2phn.get(int(c),'') for c in best_seq if c!=BLANK]
    return phonemes_to_text(phonemes)


# ─────────────────────────────────────────────────────────────
#  WER + EVALUASI
# ─────────────────────────────────────────────────────────────

def levenshtein(ref,hyp):
    R,H=len(ref),len(hyp)
    dp=np.zeros((R+1,H+1),dtype=int)
    dp[:,0]=np.arange(R+1); dp[0,:]=np.arange(H+1)
    for i in range(1,R+1):
        for j in range(1,H+1):
            c=0 if ref[i-1]==hyp[j-1] else 1
            dp[i,j]=min(dp[i-1,j]+1,dp[i,j-1]+1,dp[i-1,j-1]+c)
    i,j=R,H; S=D=I=0
    while i>0 or j>0:
        if i>0 and j>0:
            c=0 if ref[i-1]==hyp[j-1] else 1
            if dp[i,j]==dp[i-1,j-1]+c:
                if c: S+=1; i-=1;j-=1
            elif dp[i,j]==dp[i-1,j]+1: D+=1;i-=1
            else: I+=1;j-=1
        elif i>0: D+=1;i-=1
        else: I+=1;j-=1
    return min((S+D+I)/max(R,1),1.0),S,D,I,max(R,1)

def evaluate(model, eval_ids, sequences, word_refs, id2phn, split_name="eval"):
    model.set_training(False)
    print(f"\n  Mendekode {len(eval_ids)} audio [{split_name}] ...")
    wer_rows=[]; tS=tD=tI=tN=0
    for aid in eval_ids:
        lp=model.get_log_probs(sequences[aid])
        decoded=ctc_beam_decode(lp,id2phn,beam_width=BEAM_WIDTH)
        hw=decoded.split(); rw=word_refs.get(aid,[])
        wv,S,D,I,N=levenshtein(rw,hw)
        tS+=S;tD+=D;tI+=I;tN+=N
        wer_rows.append({"audio_id":aid,"WER":round(wv*100,4),
                         "S":S,"D":D,"I":I,"N":N,
                         "referensi":" ".join(rw),"hipotesis":decoded})
    wc=(tS+tD+tI)/max(tN,1)*100
    wl=[r["WER"] for r in wer_rows]
    s={"split":split_name,"n_audio":len(eval_ids),
       "wer_corpus":round(wc,4),"wer_mean":round(float(np.mean(wl)),4),
       "wer_min":round(float(np.min(wl)),4),"wer_max":round(float(np.max(wl)),4),
       "total_S":tS,"total_D":tD,"total_I":tI,"total_N":tN}
    print(f"\n── Evaluasi TDNN-CTC v5b [{split_name}] ─────────────")
    print(f"  WER Corpus : {s['wer_corpus']:.2f}%  |  Mean : {s['wer_mean']:.2f}%")
    print(f"  Min : {s['wer_min']:.2f}%  |  Max : {s['wer_max']:.2f}%")
    print(f"  S={s['total_S']}  D={s['total_D']}  I={s['total_I']}  N={s['total_N']}")
    return wer_rows, s

def print_comparison(s_test):
    g={"wer_corpus":99.84,"wer_mean":99.82}
    print(f"\n{'='*60}")
    print(f"  GMM vs TDNN-CTC v4 vs v5b")
    print(f"{'='*60}")
    print(f"  {'Metrik':<28} {'GMM':>8} {'v4 (ep1)':>10} {'v5b':>10}")
    print(f"  {'-'*58}")
    # v4 ep1 baseline
    for lbl,gv,v4v,v5v in [
        ("WER Corpus (%)", f"{g['wer_corpus']:.2f}", "~masih turun", f"{s_test['wer_corpus']:.2f}"),
        ("WER Mean (%)",   f"{g['wer_mean']:.2f}",  "~masih turun", f"{s_test['wer_mean']:.2f}"),
    ]:
        print(f"  {lbl:<28} {gv:>8} {v4v:>10} {v5v:>10}")
    print(f"{'='*60}")
    d=g["wer_corpus"]-s_test["wer_corpus"]
    if d>0: print(f"  ✓ v5b lebih baik {d:.2f}% dari GMM")
    elif d<0: print(f"  ✗ v5b lebih buruk {abs(d):.2f}% dari GMM")
    else: print(f"  = WER sama dengan GMM")


# ─────────────────────────────────────────────────────────────
#  MAIN
# ─────────────────────────────────────────────────────────────

def main():
    np.random.seed(RANDOM_SEED)

    print("="*62)
    print("  TAHAP 1 — Memuat Data & Ekstraksi MFCC")
    print("="*62)
    print(f"  Output : {DIR_SAVE}")

    (sequences_raw, targets, word_refs,
     phn2id, id2phn, vocab, actual_dim) = load_data(CSV_PATH)

    all_ids=list(sequences_raw.keys())
    train_ids,val_ids,test_ids=split_dataset(all_ids)

    mean_g,std_g=compute_global_norm(sequences_raw,train_ids)
    np.save(NORM_PATH,{"mean":mean_g,"std":std_g})
    sequences=apply_global_norm(sequences_raw,mean_g,std_g)

    print("\n"+"="*62)
    print("  TAHAP 2 — Bangun & Latih Model TDNN-CTC v5b")
    print("="*62)
    print(f"  Input dim : {actual_dim}")
    for i,(h,ctx) in enumerate(zip(HIDDEN_DIMS,TDNN_CONTEXTS)):
        print(f"  Layer {i+1}    : {h} unit  ctx={ctx}  LayerNorm=True")
    print(f"  Output    : {len(vocab)} (fonem+blank)  LayerNorm=False [BUG A fix]")
    print(f"  Dropout   : {DROPOUT_RATE}  |  Beam : {BEAM_WIDTH}")

    model=TDNNModel(
        input_dim=actual_dim,hidden_dims=HIDDEN_DIMS,
        contexts=TDNN_CONTEXTS,output_dim=len(vocab),
        dropout=DROPOUT_RATE,lr=LR_WARMUP,
    )

    history=train(model,train_ids,val_ids,sequences,targets,word_refs,id2phn)
    pd.DataFrame(history).to_csv(OUT_HISTORY_CSV,index=False)

    if os.path.exists(MODEL_PATH): model.load(MODEL_PATH)

    print("\n"+"="*62)
    print("  TAHAP 3 — Evaluasi WER")
    print("="*62)
    wer_rows_val, _   = evaluate(model,val_ids, sequences,word_refs,id2phn,"val")
    wer_rows_test,s_t = evaluate(model,test_ids,sequences,word_refs,id2phn,"test")

    all_rows=wer_rows_val+wer_rows_test
    df_w=pd.DataFrame(all_rows)
    df_w[["audio_id","WER","S","D","I","N"]].to_csv(OUT_WER_CSV,index=False)
    df_w[["audio_id","referensi","hipotesis","WER"]].to_csv(OUT_DEC_CSV,index=False)
    print(f"\n  Tersimpan:\n  {OUT_WER_CSV}\n  {OUT_DEC_CSV}")

    print("\n"+"="*62)
    print("  TAHAP 4 — Perbandingan")
    print("="*62)
    print_comparison(s_t)

    return model,s_t,history

if __name__=="__main__":
    model,summary,history=main()

In [ ]:
"""
============================================================
  TDNN + CTC v6 — Javanese ASR (Phoneme-level)
  Perbandingan dengan GMM pipeline
============================================================
  PERBAIKAN dari v5b:
  ─────────────────────────────────────────────────────────
  FIX 1 — WER ditampilkan SETIAP epoch + diagnosis blank%
    v5b: WER hanya di epoch 1,5,10,...
    v6  : WER + blank% setiap epoch → deteksi dini blank collapse

  FIX 2 — Vocab fonem dibersihkan ke 26 fonem standar Jawa
    v5b: 34 fonem (è,ê,â masuk karena regex longgar)
    v6  : normalize_text() peta è→e, ê→e, â→a, â→a
          → vocab ~26 fonem, model lebih mudah konvergen

  FIX 3 — Normalisasi MFCC: clip setelah normalisasi global
    v5b: std range [1.09, 72.84] → outlier tidak terkontrol
    v6  : np.clip(feat, -10, 10) setelah (x-mean)/std
          → gradient lebih stabil, tidak ada feature dominance

  FIX 4 — Context layer 2: [-1,2] → [-2,0] (sesuai standar Kaldi)
    v5b: [-2,2], [-1,2], [-3,3], [-7,2], [0,0]
    v6  : [-2,2], [-2,0], [-3,3], [-7,2], [0,0]
    Alasan: Layer 2 harus fokus ke history (masa lalu),
            bukan future, setelah layer 1 sudah ambil ±2

  FIX 5 — Blank collapse detection per-epoch
    Hitung % frame dengan prediksi blank di sample val
    Jika >95% → model collapse, training tidak produktif
  ─────────────────────────────────────────────────────────
"""

import os, re, time, warnings
import numpy as np
import pandas as pd
from scipy.special import logsumexp
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────
#  KONFIGURASI
# ─────────────────────────────────────────────────────────────

SAMPLED_ROOT = r"D:\doc\2026\FINAL_CODE\data\sampled_small"
CSV_PATH = (
    r"D:/doc/2026/FINAL_CODE/src/codeStat/preModelling/"
    r"samplingStratified/out_samplingStratified/dataSampledSmall.csv"
)
DIR_SAVE = (
    r"D:/doc/2026/FINAL_CODE/src/codeStat/modelling/"
    r"voiceModelling/srcVoiceModelling/TDNN/out/test_5"
)
os.makedirs(DIR_SAVE, exist_ok=True)

OUT_WER_CSV     = os.path.join(DIR_SAVE, "HasilWER_TDNN.csv")
OUT_DEC_CSV     = os.path.join(DIR_SAVE, "HasilDekode_TDNN.csv")
OUT_HISTORY_CSV = os.path.join(DIR_SAVE, "HasilTraining_TDNN.csv")
MODEL_PATH      = os.path.join(DIR_SAVE, "tdnn_best.npy")
NORM_PATH       = os.path.join(DIR_SAVE, "mfcc_global_norm.npy")

COL_PATH_DATA = "Path_Data"
COL_SENTENCES = "sentences"

# ── MFCC ──
SR=16000; N_MFCC=13; N_FFT=512; HOP_LENGTH=160
WIN_LENGTH=400; N_MELS=26; PRE_EMPH=0.97
FMIN=0.0; FMAX=4000.0; DELTA_WIDTH=2
INPUT_DIM = N_MFCC * 3   # 39
FEAT_CLIP = 10.0          # FIX 3: clip setelah normalisasi

# ── Arsitektur TDNN (FIX 4: layer 2 = [-2,0]) ──
# Referensi: Peddinti et al. 2015 "A Time Delay Neural Network
# Architecture for Efficient Modeling of Long Temporal Contexts"
# Layer 1: input, context lokal ±2 frame
# Layer 2: history focus [-2,0] — BUKAN [-1,2]
# Layer 3: wide context ±3 frame
# Layer 4: very long history [-7,2]
# Layer 5: bottleneck [0,0]
TDNN_CONTEXTS = [[-2,2], [-2,0], [-3,3], [-7,2], [0,0]]
HIDDEN_DIMS   = [256, 256, 256, 256, 256]

DROPOUT_RATE  = 0.2
LR_INIT       = 1e-3
LR_WARMUP     = 1e-5
GRAD_CLIP     = 1.0
BEAM_WIDTH    = 10

BATCH_SIZE   = 16
MAX_EPOCHS   = 80         # CTC butuh banyak epoch
EARLY_STOP   = 15         # lebih sabar
WARMUP_EPOCH = 5
LR_DECAY     = 0.5
LR_PATIENCE  = 8          # lebih sabar sebelum decay
LR_MIN       = 1e-6

TRAIN_RATIO = 0.80
VAL_RATIO   = 0.10
RANDOM_SEED = 42
BLANK       = 0


# ─────────────────────────────────────────────────────────────
#  FIX 2: NORMALISASI TEKS + G2P (vocab bersih 26 fonem)
# ─────────────────────────────────────────────────────────────

# Peta karakter bervariant ke fonem standar Bahasa Jawa
# Mengurangi vocab dari 34 → ~26 fonem
_CHAR_MAP = {
    'é': 'e', 'è': 'e', 'ê': 'e',   # semua variasi e → e
    'â': 'a', 'à': 'a', 'á': 'a',   # semua variasi a → a
    'ó': 'o', 'ô': 'o',              # variasi o → o
    'í': 'i', 'î': 'i',              # variasi i → i
    'ú': 'u', 'û': 'u',              # variasi u → u
}

#input normalisasi angka 
_ANGKA_JAWA = {
    0:'nol',1:'siji',2:'loro',3:'telu',4:'papat',
    5:'lima',6:'enem',7:'pitu',8:'wolu',9:'songo',
    10:'sepuluh',11:'sewelas',12:'rolas',
    20:'rongpuluh',100:'satus',1000:'sewu',
}

def _angka_ke_jawa(n: int) -> str:
    if n in _ANGKA_JAWA: return _ANGKA_JAWA[n]
    if n < 100:
        p = _ANGKA_JAWA.get(n//10*10, str(n//10)+'puluh')
        s = n % 10
        return p if s==0 else p + _ANGKA_JAWA.get(s, str(s))
    if n < 1000:
        r = n//100; sisa = n%100
        pfx = "satus" if r==1 else str(r)+"atus"
        return pfx if sisa==0 else pfx+_angka_ke_jawa(sisa)
    r = n//1000; sisa = n%1000
    pfx = "sewu" if r==1 else str(r)+"ewu"
    return pfx if sisa==0 else pfx+_angka_ke_jawa(sisa)


def normalize_text(text: str) -> str:
    """
    Normalisasi teks sebelum G2P:
    1. Lowercase
    2. Angka → kata Jawa
    3. Peta karakter bervariant → standar (FIX 2)
    4. Hapus karakter non-alfabet
    5. Normalkan spasi
    """
    text = text.lower().strip()
    # Angka → kata
    text = re.sub(r'\d+',
                  lambda m: ' ' + _angka_ke_jawa(int(m.group())) + ' ',
                  text)
    # FIX 2: peta karakter bervariant
    result = []
    for ch in text:
        result.append(_CHAR_MAP.get(ch, ch))
    text = ''.join(result)
    # Hanya huruf a-z dan spasi
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


# Digraf Bahasa Jawa
DIGRAPHS = {'ng': 'ŋ', 'ny': 'ɲ', 'th': 'tʰ', 'dh': 'dʰ'}


def javanese_g2p(text: str) -> list:
    """
    Grapheme-to-Phoneme untuk Bahasa Jawa.
    Input sudah dinormalisasi (hanya a-z + spasi).

    Rules:
      ng → /ŋ/   (velar nasal, e.g. "ngajeng", "menyang")
      ny → /ɲ/   (palatal nasal, e.g. "nyuwun", "banyu")
      th → /tʰ/  (dental aspirat, e.g. "thole")
      dh → /dʰ/  (dental bersuara, e.g. "dhahar")
      Karakter tunggal → fonem langsung (Jawa hampir fonemis)
      Spasi → <sp> (silence antar kata)
    """
    text     = normalize_text(text)
    phonemes = []
    i        = 0

    while i < len(text):
        ch = text[i]

        if ch == ' ':
            if phonemes and phonemes[-1] != '<sp>':
                phonemes.append('<sp>')
            i += 1
            continue

        if not ch.isalpha():
            i += 1
            continue

        # Cek digraf
        if i + 1 < len(text):
            two = text[i:i+2]
            if two in DIGRAPHS:
                phonemes.append(DIGRAPHS[two])
                i += 2
                continue

        phonemes.append(ch)
        i += 1

    return phonemes


def build_phoneme_vocab(sentences: list) -> tuple:
    """Bangun vocab dari seluruh dataset. Index 0 = <blank>."""
    all_phn = set()
    for s in sentences:
        all_phn.update(javanese_g2p(s))
    vocab   = ['<blank>'] + sorted(all_phn)
    phn2id  = {p: i for i, p in enumerate(vocab)}
    id2phn  = {i: p for p, i in phn2id.items()}
    return vocab, phn2id, id2phn


def encode_phoneme(sentence: str, phn2id: dict) -> list:
    return [phn2id[p] for p in javanese_g2p(sentence) if p in phn2id]


def phonemes_to_text(phonemes: list) -> str:
    """Inverse G2P: fonem sequence → teks."""
    result = []
    for p in phonemes:
        if   p == '<sp>': result.append(' ')
        elif p == 'ŋ':    result.append('ng')
        elif p == 'ɲ':    result.append('ny')
        elif p == 'tʰ':   result.append('th')
        elif p == 'dʰ':   result.append('dh')
        else:             result.append(p)
    return ''.join(result).strip()


# ─────────────────────────────────────────────────────────────
#  EKSTRAKSI MFCC (identik GMM)
# ─────────────────────────────────────────────────────────────

try:
    import librosa; LIBROSA_OK = True
except ImportError:
    LIBROSA_OK = False
try:
    from scipy.io import wavfile; SCIPY_OK = True
except ImportError:
    SCIPY_OK = False


def extract_features_raw(wav_path: str) -> np.ndarray:
    """Ekstrak MFCC 39-dim raw (belum dinormalisasi global)."""
    if not os.path.isfile(wav_path):
        raise FileNotFoundError(wav_path)
    return _librosa_raw(wav_path) if LIBROSA_OK else _manual_raw(wav_path)


def _librosa_raw(wav_path):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    if len(y) == 0: raise ValueError("Audio kosong")
    y    = np.append(y[0], y[1:] - PRE_EMPH * y[:-1])
    mfcc = librosa.feature.mfcc(
        y=y, sr=SR, n_mfcc=N_MFCC, n_fft=N_FFT,
        hop_length=HOP_LENGTH, win_length=WIN_LENGTH,
        n_mels=N_MELS, fmin=FMIN, fmax=FMAX, window="hamming"
    )
    d1   = librosa.feature.delta(mfcc, width=DELTA_WIDTH*2+1)
    d2   = librosa.feature.delta(mfcc, order=2, width=DELTA_WIDTH*2+1)
    return np.vstack([mfcc, d1, d2]).T.astype(np.float32)


def _delta_vec(feat, width=2):
    T=feat.shape[0]; denom=2.0*sum(n*n for n in range(1,width+1))
    delta=np.zeros_like(feat)
    for n in range(1,width+1):
        delta += n*(feat[np.clip(np.arange(T)+n,0,T-1)] -
                    feat[np.clip(np.arange(T)-n,0,T-1)])
    return delta/denom


def _manual_raw(wav_path):
    if not SCIPY_OK: raise ImportError("Butuh librosa atau scipy")
    sr, audio = wavfile.read(wav_path)
    if audio.dtype==np.int16: audio=audio/32768.0
    elif audio.dtype==np.int32: audio=audio/2147483648.0
    audio=audio.astype(np.float32)
    if sr!=SR:
        n=int(len(audio)*SR/sr)
        audio=np.interp(np.linspace(0,len(audio)-1,n),
                        np.arange(len(audio)),audio)
    y=np.append(audio[0],audio[1:]-PRE_EMPH*audio[:-1])
    T_fr=1+(len(y)-WIN_LENGTH)//HOP_LENGTH
    idx=np.arange(WIN_LENGTH)[None,:]+np.arange(T_fr)[:,None]*HOP_LENGTH
    frames=y[idx]*np.hamming(WIN_LENGTH)
    fft_m=np.abs(np.fft.rfft(frames,n=N_FFT))
    def hz2mel(f): return 1127*np.log1p(f/700)
    def mel2hz(m): return 700*(np.exp(m/1127)-1)
    pts=mel2hz(np.linspace(hz2mel(FMIN),hz2mel(FMAX),N_MELS+2))
    bins=np.floor((N_FFT+1)*pts/SR).astype(int)
    fb=np.zeros((N_MELS,N_FFT//2+1))
    for m in range(1,N_MELS+1):
        for k in range(bins[m-1],bins[m]):
            fb[m-1,k]=(k-bins[m-1])/(bins[m]-bins[m-1]+1e-10)
        for k in range(bins[m],bins[m+1]):
            fb[m-1,k]=(bins[m+1]-k)/(bins[m+1]-bins[m]+1e-10)
    log_mel=np.log10(np.maximum(fft_m**2@fb.T,1e-10))
    N_=N_MELS; ni=np.arange(N_MFCC)[:,None]; ki=np.arange(N_)[None,:]+0.5
    mfcc=log_mel@(np.cos(np.pi*ni*ki/N_)*np.sqrt(2/N_)).T
    d1=_delta_vec(mfcc,DELTA_WIDTH); d2=_delta_vec(d1,DELTA_WIDTH)
    return np.concatenate([mfcc,d1,d2],axis=1).astype(np.float32)


# ─────────────────────────────────────────────────────────────
#  NORMALISASI MFCC GLOBAL + CLIP (FIX 3)
# ─────────────────────────────────────────────────────────────

def compute_global_norm(sequences: dict, train_ids: list) -> tuple:
    """Hitung mean & std dari train set saja (bukan seluruh data)."""
    print("  Menghitung normalisasi global dari train set ...")
    all_frames = np.concatenate([sequences[a] for a in train_ids], axis=0)
    mean = all_frames.mean(axis=0).astype(np.float32)
    std  = (all_frames.std(axis=0) + 1e-8).astype(np.float32)
    print(f"  mean [{mean.min():.2f}, {mean.max():.2f}]  "
          f"std [{std.min():.4f}, {std.max():.2f}]")
    return mean, std


def apply_global_norm(sequences: dict, mean: np.ndarray,
                      std: np.ndarray) -> dict:
    """
    Normalisasi + clip (FIX 3).
    Clip ke [-10, 10] mencegah outlier mendominasi gradient.
    """
    normed = {}
    for aid, feat in sequences.items():
        f = (feat - mean) / std
        f = np.clip(f, -FEAT_CLIP, FEAT_CLIP)   # FIX 3
        normed[aid] = f.astype(np.float32)
    return normed


# ─────────────────────────────────────────────────────────────
#  DATA LOADER
# ─────────────────────────────────────────────────────────────

def find_wav(audio_id: str) -> str:
    for root, _, files in os.walk(SAMPLED_ROOT):
        for f in files:
            if (os.path.splitext(f)[0] == audio_id and
                    f.lower().endswith(".wav")):
                return os.path.join(root, f)
    return ""


def load_data(csv_path: str) -> tuple:
    print(f"  Membaca {csv_path} ...")
    df = pd.read_csv(csv_path)
    print(f"  Shape CSV : {df.shape}")

    for col in [COL_PATH_DATA, COL_SENTENCES]:
        if col not in df.columns:
            raise ValueError(f"Kolom '{col}' tidak ada.")

    # Normalisasi teks dan bangun vocab
    sentences_norm = [normalize_text(str(s))
                      for s in df[COL_SENTENCES].tolist()]
    vocab, phn2id, id2phn = build_phoneme_vocab(sentences_norm)
    print(f"  Vocab fonem : {len(vocab)-1}  (+1 blank → total {len(vocab)})")
    print(f"  Fonem       : {sorted(set(vocab) - {'<blank>'})}")

    sequences_raw, targets, word_refs = {}, {}, {}
    skipped = 0

    print(f"  Mengekstrak MFCC dari {len(df)} file .wav ...")
    for idx, row in df.iterrows():
        audio_id = os.path.splitext(
            os.path.basename(str(row[COL_PATH_DATA])))[0]
        kalimat  = normalize_text(str(row[COL_SENTENCES]))
        words    = kalimat.split()

        wav_path = str(row[COL_PATH_DATA])
        if not os.path.isfile(wav_path):
            wav_path = find_wav(audio_id)
        if not wav_path:
            skipped += 1; continue

        try:
            feats  = extract_features_raw(wav_path)
            T      = feats.shape[0]
            target = encode_phoneme(kalimat, phn2id)
            if T == 0 or not target:
                skipped += 1; continue
            if T < 2 * len(target) + 1:
                skipped += 1; continue
            sequences_raw[audio_id] = feats
            targets[audio_id]       = target
            word_refs[audio_id]     = words
        except Exception as e:
            print(f"  ⚠ {audio_id}: {e}")
            skipped += 1; continue

        if (idx+1) % 200 == 0 or (idx+1) == len(df):
            print(f"    [{idx+1}/{len(df)}] diproses, {skipped} dilewati")

    actual_dim = next(iter(sequences_raw.values())).shape[1]
    lens = [len(targets[a]) for a in sequences_raw]
    print(f"\n  Audio berhasil : {len(sequences_raw)} | dilewati: {skipped}")
    print(f"  Panjang fonem  : min={min(lens)}, "
          f"mean={np.mean(lens):.1f}, max={max(lens)}")
    return sequences_raw, targets, word_refs, phn2id, id2phn, vocab, actual_dim


# ─────────────────────────────────────────────────────────────
#  CTC LOSS + GRADIENT (vectorized, log-space)
# ─────────────────────────────────────────────────────────────

def _softmax(logits: np.ndarray) -> np.ndarray:
    x = logits - logits.max(axis=1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=1, keepdims=True)


def ctc_loss_and_grad(logits: np.ndarray, targets: list) -> tuple:
    T, V = logits.shape
    L    = len(targets)
    S    = 2 * L + 1
    NEG  = -1e9

    l_ = np.zeros(S, dtype=np.int32)
    for i, c in enumerate(targets):
        l_[2*i+1] = c

    probs     = _softmax(logits).astype(np.float64)
    log_probs = np.log(probs + 1e-30)
    lp_ext    = log_probs[:, l_]

    # Precompute transition masks
    can_skip = np.zeros(S, dtype=bool)
    for s in range(2, S):
        if l_[s] != BLANK and l_[s] != l_[s-2]:
            can_skip[s] = True
    can_skip_bk = np.zeros(S, dtype=bool)
    for s in range(S-2):
        if l_[s+2] != BLANK and l_[s+2] != l_[s]:
            can_skip_bk[s] = True

    # Alpha (forward)
    alpha = np.full((T, S), NEG)
    alpha[0, 0] = lp_ext[0, 0]
    if S > 1: alpha[0, 1] = lp_ext[0, 1]
    for t in range(1, T):
        s0 = alpha[t-1]
        s1 = np.empty(S); s1[0]=NEG; s1[1:]=alpha[t-1,:-1]
        s2 = np.empty(S); s2[:2]=NEG; s2[2:]=alpha[t-1,:-2]
        base    = np.logaddexp(s0, s1)
        alpha[t] = np.where(can_skip, np.logaddexp(base, s2), base) + lp_ext[t]

    # Beta (backward)
    beta = np.full((T, S), NEG)
    beta[T-1, S-1] = 0.0
    if S > 1: beta[T-1, S-2] = 0.0
    for t in range(T-2, -1, -1):
        lp  = lp_ext[t+1]
        s0  = beta[t+1] + lp
        s1  = np.empty(S); s1[-1]=NEG; s1[:-1]=beta[t+1,1:]+lp[1:]
        s2  = np.empty(S); s2[-2:]=NEG; s2[:-2]=beta[t+1,2:]+lp[2:]
        base    = np.logaddexp(s0, s1)
        beta[t] = np.where(can_skip_bk, np.logaddexp(base, s2), base)

    # Log-likelihood
    ll_terms = [alpha[T-1, S-1]]
    if S > 1: ll_terms.append(alpha[T-1, S-2])
    ll = float(logsumexp(ll_terms))
    if not np.isfinite(ll) or ll < -1e7:
        return 0.0, np.zeros((T, V), dtype=np.float32)

    loss = float(-ll / T)

    # Gradient log-space
    ab        = alpha + beta
    log_numer = np.full((T, V), NEG)
    for s in range(S):
        log_numer[:, l_[s]] = np.logaddexp(log_numer[:, l_[s]], ab[:, s])
    term = np.exp(np.clip(log_numer - ll, -500, 0))
    return loss, (probs - term).astype(np.float32)


# ─────────────────────────────────────────────────────────────
#  LAYER NORM
# ─────────────────────────────────────────────────────────────

class LayerNorm:
    def __init__(self, dim, lr=1e-3, eps=1e-5):
        self.eps   = eps
        self.lr    = lr
        self.gamma = np.ones(dim,  dtype=np.float32)
        self.beta  = np.zeros(dim, dtype=np.float32)
        self.mg=np.zeros(dim,dtype=np.float32)
        self.vg=np.zeros(dim,dtype=np.float32)
        self.mb=np.zeros(dim,dtype=np.float32)
        self.vb=np.zeros(dim,dtype=np.float32)
        self.t=0
        self._xhat=None; self._std=None
        self._dg_acc=np.zeros(dim,dtype=np.float32)
        self._db_acc=np.zeros(dim,dtype=np.float32)
        self._n_acc=0

    def forward(self, x):
        mean = x.mean(axis=1, keepdims=True)
        std  = np.sqrt(x.var(axis=1, keepdims=True) + self.eps)
        xhat = (x - mean) / std
        self._xhat = xhat; self._std = std
        return self.gamma * xhat + self.beta

    def backward(self, dout, accumulate=True):
        T, D = dout.shape
        dg   = (dout * self._xhat).sum(axis=0)
        db   = dout.sum(axis=0)
        dxhat = dout * self.gamma
        dx = (1.0/D/self._std) * (
            D*dxhat
            - dxhat.sum(axis=1, keepdims=True)
            - self._xhat * (dxhat*self._xhat).sum(axis=1, keepdims=True)
        )
        if accumulate:
            self._dg_acc+=dg; self._db_acc+=db; self._n_acc+=1
        else:
            self._adam(dg, db)
        return dx

    def flush(self):
        if self._n_acc == 0: return
        self._adam(self._dg_acc/self._n_acc, self._db_acc/self._n_acc)
        self._dg_acc[:]=0; self._db_acc[:]=0; self._n_acc=0

    def set_lr(self, lr): self.lr = lr

    def _adam(self, dg, db, b1=0.9, b2=0.999, eps=1e-8):
        self.t+=1
        self.mg=b1*self.mg+(1-b1)*dg; self.vg=b2*self.vg+(1-b2)*dg**2
        self.gamma-=self.lr*(self.mg/(1-b1**self.t))/(np.sqrt(self.vg/(1-b2**self.t))+eps)
        self.mb=b1*self.mb+(1-b1)*db; self.vb=b2*self.vb+(1-b2)*db**2
        self.beta -=self.lr*(self.mb/(1-b1**self.t))/(np.sqrt(self.vb/(1-b2**self.t))+eps)


# ─────────────────────────────────────────────────────────────
#  TDNN LAYER
# ─────────────────────────────────────────────────────────────

class TDNNLayer:
    def __init__(self, in_dim, out_dim, context, lr, use_ln=True):
        self.context = context
        self.ctx_len = context[1] - context[0] + 1
        self.lr      = lr
        self.use_ln  = use_ln
        fan_in       = in_dim * self.ctx_len
        self.W = (np.random.randn(fan_in, out_dim) *
                  np.sqrt(2.0/fan_in)).astype(np.float32)
        self.b = np.zeros(out_dim, dtype=np.float32)
        if use_ln:
            self.ln = LayerNorm(out_dim, lr=lr)
        self.mW=np.zeros_like(self.W); self.vW=np.zeros_like(self.W)
        self.mb=np.zeros_like(self.b); self.vb=np.zeros_like(self.b)
        self.t=0; self._X=None
        self._dW_acc=np.zeros_like(self.W)
        self._db_acc=np.zeros_like(self.b)
        self._n_acc=0

    def splice(self, X: np.ndarray) -> np.ndarray:
        T=X.shape[0]; idx=np.arange(T)
        return np.concatenate(
            [X[np.clip(idx+off,0,T-1)]
             for off in range(self.context[0],self.context[1]+1)],
            axis=1)

    def forward(self, Xsp: np.ndarray) -> np.ndarray:
        self._X = Xsp
        z = Xsp @ self.W + self.b
        if self.use_ln: z = self.ln.forward(z)
        return z

    def backward(self, dout: np.ndarray, accumulate=True) -> np.ndarray:
        T = len(dout)
        if self.use_ln: dout = self.ln.backward(dout, accumulate=accumulate)
        dW   = self._X.T @ dout / T
        db   = dout.mean(0)
        dXsp = dout @ self.W.T
        if accumulate:
            self._dW_acc+=dW; self._db_acc+=db; self._n_acc+=1
        else:
            self._adam(dW, db)
        B, D_tot = dXsp.shape; D = D_tot // self.ctx_len
        dX = np.zeros((B,D), dtype=np.float32)
        for k in range(self.ctx_len):
            dX += dXsp[:, k*D:(k+1)*D]
        return dX

    def flush(self):
        if self._n_acc == 0: return
        self._adam(self._dW_acc/self._n_acc, self._db_acc/self._n_acc)
        if self.use_ln: self.ln.flush()
        self._dW_acc[:]=0; self._db_acc[:]=0; self._n_acc=0

    def set_lr(self, lr):
        self.lr = lr
        if self.use_ln: self.ln.set_lr(lr)

    def _adam(self, dW, db, b1=0.9, b2=0.999, eps=1e-8):
        self.t+=1
        self.mW=b1*self.mW+(1-b1)*dW; self.vW=b2*self.vW+(1-b2)*dW**2
        self.W-=self.lr*(self.mW/(1-b1**self.t))/(np.sqrt(self.vW/(1-b2**self.t))+eps)
        self.mb=b1*self.mb+(1-b1)*db; self.vb=b2*self.vb+(1-b2)*db**2
        self.b-=self.lr*(self.mb/(1-b1**self.t))/(np.sqrt(self.vb/(1-b2**self.t))+eps)


# ─────────────────────────────────────────────────────────────
#  MODEL TDNN
# ─────────────────────────────────────────────────────────────

class TDNNModel:
    def __init__(self, input_dim, hidden_dims, contexts,
                 output_dim, dropout=0.2, lr=1e-3):
        assert len(hidden_dims) == len(contexts)
        self.dropout  = dropout
        self.training = True
        self.layers   = []
        in_d = input_dim
        for h, ctx in zip(hidden_dims, contexts):
            self.layers.append(TDNNLayer(in_d, h, ctx, lr, use_ln=True))
            in_d = h
        # Output layer: tanpa LayerNorm
        self.layers.append(TDNNLayer(in_d, output_dim, [0,0], lr, use_ln=False))
        self._rc=[]; self._dc=[]

    def set_training(self, mode: bool):
        self.training = mode

    def set_lr(self, lr: float):
        for layer in self.layers: layer.set_lr(lr)

    def forward(self, X: np.ndarray) -> np.ndarray:
        self._rc=[]; self._dc=[]; h=X
        for layer in self.layers[:-1]:
            z = layer.forward(layer.splice(h))
            a = np.maximum(0.0, z)
            if self.training and self.dropout > 0:
                mask = (np.random.rand(*a.shape) > self.dropout
                        ).astype(np.float32)
                a    = a * mask / (1.0 - self.dropout)
            else:
                mask = np.ones(a.shape, dtype=np.float32)
            self._rc.append(z); self._dc.append(mask); h=a
        out = self.layers[-1]
        return out.forward(out.splice(h))

    def ctc_step(self, X: np.ndarray, target: list,
                 accumulate=True) -> float:
        logits       = self.forward(X)
        loss, dlogit = ctc_loss_and_grad(logits, target)
        if loss == 0.0: return 0.0
        gnorm = np.sqrt((dlogit**2).sum())
        if gnorm > GRAD_CLIP: dlogit = dlogit*(GRAD_CLIP/gnorm)
        sc = 1.0/(1.0-self.dropout+1e-9) if self.dropout > 0 else 1.0
        dh = self.layers[-1].backward(dlogit, accumulate=accumulate)
        for i in reversed(range(len(self.layers)-1)):
            dh = dh * self._dc[i] * sc
            dh = dh * (self._rc[i] > 0).astype(np.float32)
            dh = self.layers[i].backward(dh, accumulate=accumulate)
        return loss

    def flush(self):
        for layer in self.layers: layer.flush()

    def get_log_probs(self, X: np.ndarray) -> np.ndarray:
        self.set_training(False)
        p = _softmax(self.forward(X))
        return np.log(p + 1e-30)

    def save(self, path: str):
        data = {
            "w" : [(l.W.copy(), l.b.copy()) for l in self.layers],
            "ln": [(l.ln.gamma.copy(), l.ln.beta.copy())
                   if l.use_ln else (None, None)
                   for l in self.layers],
        }
        np.save(path, data, allow_pickle=True)

    def load(self, path: str):
        d = np.load(path, allow_pickle=True).item()
        for i, (W, b) in enumerate(d["w"]):
            self.layers[i].W = W; self.layers[i].b = b
        if "ln" in d:
            for i, (g, bt) in enumerate(d["ln"]):
                if g is not None and self.layers[i].use_ln:
                    self.layers[i].ln.gamma = g
                    self.layers[i].ln.beta  = bt
        print(f"  Model dimuat: {path}")


# ─────────────────────────────────────────────────────────────
#  SPLIT
# ─────────────────────────────────────────────────────────────

def split_dataset(ids):
    rng  = np.random.default_rng(RANDOM_SEED)
    ids  = list(ids); perm = rng.permutation(len(ids))
    n    = len(ids); ntr = int(n*TRAIN_RATIO); nva = int(n*VAL_RATIO)
    tr   = [ids[i] for i in perm[:ntr]]
    va   = [ids[i] for i in perm[ntr:ntr+nva]]
    te   = [ids[i] for i in perm[ntr+nva:]]
    print(f"  Split → train:{len(tr)}  val:{len(va)}  test:{len(te)}")
    return tr, va, te


# ─────────────────────────────────────────────────────────────
#  BEAM SEARCH CTC DECODE
# ─────────────────────────────────────────────────────────────

def ctc_beam_decode(log_probs: np.ndarray, id2phn: dict,
                    beam_width: int = 10) -> str:
    T, V  = log_probs.shape
    NEG   = -float('inf')
    # beam: seq → (log_Pb, log_Pnb)
    beam  = {(): (0.0, NEG)}

    for t in range(T):
        new_beam = {}
        for seq, (pb, pnb) in beam.items():
            # Extend dengan blank
            lp_b = log_probs[t, BLANK]
            p0, p1 = new_beam.get(seq, (NEG, NEG))
            new_beam[seq] = (np.logaddexp(p0, np.logaddexp(pb,pnb)+lp_b), p1)

            # Extend dengan setiap non-blank
            for c in range(1, V):
                lp_c  = log_probs[t, c]
                nseq  = seq + (c,)
                ep_nb = (pb + lp_c if seq and seq[-1] == c
                         else np.logaddexp(pb, pnb) + lp_c)
                p0n, p1n = new_beam.get(nseq, (NEG, NEG))
                new_beam[nseq] = (p0n, np.logaddexp(p1n, ep_nb))

        # Prune ke beam_width terbaik
        beam = dict(
            sorted(new_beam.items(),
                   key=lambda kv: np.logaddexp(kv[1][0], kv[1][1]),
                   reverse=True)[:beam_width]
        )

    best_seq = max(beam.items(),
                   key=lambda kv: np.logaddexp(kv[1][0], kv[1][1]))[0]
    phonemes = [id2phn.get(int(c), '') for c in best_seq if c != BLANK]
    return phonemes_to_text(phonemes)


# ─────────────────────────────────────────────────────────────
#  FIX 5: DIAGNOSIS BLANK COLLAPSE
# ─────────────────────────────────────────────────────────────

def diagnose_blank(model: TDNNModel, sample_ids: list,
                   sequences: dict, n: int = 20) -> float:
    """
    Hitung % frame dengan prediksi blank di n sampel.
    Jika > 95% → model collapse ke blank (tidak belajar fonem).
    """
    model.set_training(False)
    total_blank = total_frame = 0
    for aid in sample_ids[:n]:
        lp    = model.get_log_probs(sequences[aid])
        preds = np.argmax(lp, axis=1)
        total_blank += (preds == BLANK).sum()
        total_frame += len(preds)
    return total_blank / max(total_frame, 1) * 100


# ─────────────────────────────────────────────────────────────
#  FIX 1: TRAINING — WER ditampilkan SETIAP epoch
# ─────────────────────────────────────────────────────────────

def compute_wer_sample(model: TDNNModel, ids: list,
                       sequences: dict, word_refs: dict,
                       id2phn: dict, n: int = 30) -> float:
    """WER pada n sampel acak untuk monitoring cepat."""
    rng    = np.random.default_rng(RANDOM_SEED)
    sample = list(rng.choice(ids, min(n, len(ids)), replace=False))
    model.set_training(False)
    tS=tD=tI=tN=0
    for aid in sample:
        lp      = model.get_log_probs(sequences[aid])
        decoded = ctc_beam_decode(lp, id2phn, beam_width=5)
        hw = decoded.split()
        rw = word_refs.get(aid, [])
        _, S, D, I, N = levenshtein(rw, hw)
        tS+=S; tD+=D; tI+=I; tN+=N
    return (tS+tD+tI) / max(tN, 1) * 100


def get_lr(epoch: int, no_improve: int) -> float:
    if epoch <= WARMUP_EPOCH:
        alpha = (epoch-1) / max(WARMUP_EPOCH-1, 1)
        return LR_WARMUP + alpha*(LR_INIT - LR_WARMUP)
    return max(LR_INIT * (LR_DECAY ** (no_improve // LR_PATIENCE)), LR_MIN)


def train(model: TDNNModel, train_ids, val_ids,
          sequences, targets, word_refs, id2phn):
    rng  = np.random.default_rng(RANDOM_SEED)
    hist = []
    best = float("inf")
    pat  = 0
    no_improve = 0

    print(f"\n  Train: {len(train_ids)} | Val: {len(val_ids)}")
    print(f"  Epochs: {MAX_EPOCHS} | EarlyStop: {EARLY_STOP} | "
          f"Batch: {BATCH_SIZE}")
    print(f"  LR: warmup→{LR_INIT} ({WARMUP_EPOCH}ep), "
          f"decay×{LR_DECAY} tiap {LR_PATIENCE}ep tanpa perbaikan")
    # FIX 1: header dengan WER dan blank% setiap epoch
    print(f"\n  {'Ep':>4}  {'LR':>8}  {'TrLoss':>8}  {'VaLoss':>8}  "
          f"{'WER%':>6}  {'Blank%':>7}  {'t':>6}")
    print(f"  {'-'*62}")

    for epoch in range(1, MAX_EPOCHS+1):
        t0 = time.time()
        lr = get_lr(epoch, no_improve)
        model.set_lr(lr)
        model.set_training(True)

        perm  = rng.permutation(len(train_ids))
        order = [train_ids[i] for i in perm]
        total_loss = 0.0; n_batches = 0

        for start in range(0, len(order), BATCH_SIZE):
            batch      = order[start:start+BATCH_SIZE]
            batch_loss = 0.0; valid = 0
            for aid in batch:
                try:
                    lv = model.ctc_step(sequences[aid], targets[aid],
                                        accumulate=True)
                    if lv > 0: batch_loss += lv; valid += 1
                except Exception: continue
            model.flush()
            if valid > 0: total_loss += batch_loss/valid; n_batches += 1

        tr_loss = total_loss / max(n_batches, 1)

        # Validasi (FIX 4 dari v5b: training=False)
        model.set_training(False)
        val_loss = 0.0; val_n = 0
        for aid in val_ids:
            try:
                logits = model.forward(sequences[aid])
                lv, _  = ctc_loss_and_grad(logits, targets[aid])
                if lv > 0: val_loss += lv; val_n += 1
            except Exception: continue
        val_loss /= max(val_n, 1)

        # FIX 1: WER dan blank% SETIAP epoch
        wer_val   = compute_wer_sample(model, val_ids, sequences,
                                        word_refs, id2phn, n=30)
        blank_pct = diagnose_blank(model, val_ids, sequences, n=20)

        elapsed = time.time() - t0
        hist.append({
            "epoch"      : epoch,
            "lr"         : round(lr, 8),
            "train_loss" : round(tr_loss, 6),
            "val_loss"   : round(val_loss, 6),
            "val_wer_pct": round(wer_val, 2),
            "blank_pct"  : round(blank_pct, 1),
        })
        print(f"  {epoch:4d}  {lr:8.6f}  {tr_loss:8.4f}  {val_loss:8.4f}  "
              f"{wer_val:6.1f}  {blank_pct:6.1f}%  {elapsed:5.0f}s")

        # FIX 5: peringatan blank collapse
        if blank_pct > 97.0 and epoch > WARMUP_EPOCH:
            print(f"  ⚠ Blank collapse ({blank_pct:.1f}%) — "
                  f"coba turunkan LR atau periksa data")

        if val_loss < best - 1e-5:
            best = val_loss; pat = 0; no_improve = 0
            model.save(MODEL_PATH)
        else:
            pat += 1; no_improve += 1
            if pat >= EARLY_STOP:
                print(f"\n  ✓ Early stop epoch {epoch} (best={best:.4f})")
                break

    return hist


# ─────────────────────────────────────────────────────────────
#  WER
# ─────────────────────────────────────────────────────────────

def levenshtein(ref: list, hyp: list) -> tuple:
    R, H = len(ref), len(hyp)
    dp   = np.zeros((R+1, H+1), dtype=int)
    dp[:,0]=np.arange(R+1); dp[0,:]=np.arange(H+1)
    for i in range(1, R+1):
        for j in range(1, H+1):
            c = 0 if ref[i-1]==hyp[j-1] else 1
            dp[i,j] = min(dp[i-1,j]+1, dp[i,j-1]+1, dp[i-1,j-1]+c)
    i, j = R, H; S=D=I=0
    while i>0 or j>0:
        if i>0 and j>0:
            c = 0 if ref[i-1]==hyp[j-1] else 1
            if dp[i,j]==dp[i-1,j-1]+c:
                if c: S+=1; i-=1; j-=1
            elif dp[i,j]==dp[i-1,j]+1: D+=1; i-=1
            else: I+=1; j-=1
        elif i>0: D+=1; i-=1
        else:     I+=1; j-=1
    return min((S+D+I)/max(R,1), 1.0), S, D, I, max(R, 1)


# ─────────────────────────────────────────────────────────────
#  EVALUASI
# ─────────────────────────────────────────────────────────────

def evaluate(model: TDNNModel, eval_ids: list,
             sequences: dict, word_refs: dict,
             id2phn: dict, split_name: str = "eval") -> tuple:
    model.set_training(False)
    print(f"\n  Mendekode {len(eval_ids)} audio [{split_name}] ...")
    wer_rows=[]; tS=tD=tI=tN=0

    for aid in eval_ids:
        lp      = model.get_log_probs(sequences[aid])
        decoded = ctc_beam_decode(lp, id2phn, beam_width=BEAM_WIDTH)
        hw = decoded.split()
        rw = word_refs.get(aid, [])
        wv, S, D, I, N = levenshtein(rw, hw)
        tS+=S; tD+=D; tI+=I; tN+=N
        wer_rows.append({
            "audio_id" : aid,
            "WER"      : round(wv*100, 4),
            "S": S, "D": D, "I": I, "N": N,
            "referensi": " ".join(rw),
            "hipotesis": decoded,
        })

    wc = (tS+tD+tI) / max(tN,1) * 100
    wl = [r["WER"] for r in wer_rows]
    s  = {
        "split"      : split_name,
        "n_audio"    : len(eval_ids),
        "wer_corpus" : round(wc, 4),
        "wer_mean"   : round(float(np.mean(wl)), 4),
        "wer_min"    : round(float(np.min(wl)), 4),
        "wer_max"    : round(float(np.max(wl)), 4),
        "total_S"    : tS, "total_D": tD,
        "total_I"    : tI, "total_N": tN,
    }

    print(f"\n── Evaluasi TDNN-CTC v6 [{split_name}] ─────────────────")
    print(f"  Jumlah audio : {s['n_audio']}")
    print(f"  WER Corpus   : {s['wer_corpus']:.2f}%")
    print(f"  WER Mean     : {s['wer_mean']:.2f}%")
    print(f"  WER Min      : {s['wer_min']:.2f}%")
    print(f"  WER Max      : {s['wer_max']:.2f}%")
    print(f"  Total S      : {s['total_S']}")
    print(f"  Total D      : {s['total_D']}")
    print(f"  Total I      : {s['total_I']}")
    print(f"  Total N(kata): {s['total_N']}")
    print(f"────────────────────────────────────────────────────────")
    return wer_rows, s


# ─────────────────────────────────────────────────────────────
#  PERBANDINGAN GMM vs TDNN
# ─────────────────────────────────────────────────────────────

def print_comparison(s_test: dict):
    g = {"wer_corpus":99.84,"wer_mean":99.82,"wer_min":50.00,
         "wer_max":100.00,"total_S":1730,"total_D":10394,
         "total_I":0,"total_N":12143}
    print(f"\n{'='*62}")
    print(f"  PERBANDINGAN GMM vs TDNN-CTC v6 (Phoneme)")
    print(f"{'='*62}")
    print(f"  {'Metrik':<30} {'GMM':>10} {'TDNN-CTC':>10}")
    print(f"  {'-'*52}")
    for lbl, gv, tv in [
        ("WER Corpus (%)",       f"{g['wer_corpus']:.2f}", f"{s_test['wer_corpus']:.2f}"),
        ("WER Mean (%)",         f"{g['wer_mean']:.2f}",   f"{s_test['wer_mean']:.2f}"),
        ("WER Min (%)",          f"{g['wer_min']:.2f}",    f"{s_test['wer_min']:.2f}"),
        ("WER Max (%)",          f"{g['wer_max']:.2f}",    f"{s_test['wer_max']:.2f}"),
        ("Total S (substitusi)", str(g['total_S']),        str(s_test['total_S'])),
        ("Total D (deletion)",   str(g['total_D']),        str(s_test['total_D'])),
        ("Total I (insertion)",  str(g['total_I']),        str(s_test['total_I'])),
        ("Total N (kata ref)",   str(g['total_N']),        str(s_test['total_N'])),
    ]:
        print(f"  {lbl:<30} {gv:>10} {tv:>10}")
    print(f"{'='*62}")
    d = g["wer_corpus"] - s_test["wer_corpus"]
    if   d > 0: print(f"  ✓ TDNN-CTC lebih baik {d:.2f}% dari GMM")
    elif d < 0: print(f"  ✗ TDNN-CTC lebih buruk {abs(d):.2f}% dari GMM")
    else:       print(f"  = WER sama dengan GMM")


# ─────────────────────────────────────────────────────────────
#  MAIN
# ─────────────────────────────────────────────────────────────

def main():
    np.random.seed(RANDOM_SEED)

    print("="*62)
    print("  TAHAP 1 — Memuat Data & Ekstraksi MFCC")
    print("="*62)
    print(f"  Folder audio : {SAMPLED_ROOT}")
    print(f"  CSV          : {CSV_PATH}")
    print(f"  Output       : {DIR_SAVE}")
    print(f"  Label        : fonem Bahasa Jawa via G2P (CTC)")

    (sequences_raw, targets, word_refs,
     phn2id, id2phn, vocab, actual_dim) = load_data(CSV_PATH)

    all_ids = list(sequences_raw.keys())
    train_ids, val_ids, test_ids = split_dataset(all_ids)

    # Normalisasi global (dari train only) + clip
    mean_g, std_g = compute_global_norm(sequences_raw, train_ids)
    np.save(NORM_PATH, {"mean": mean_g, "std": std_g})
    sequences = apply_global_norm(sequences_raw, mean_g, std_g)

    print("\n"+"="*62)
    print("  TAHAP 2 — Bangun & Latih Model TDNN-CTC v6")
    print("="*62)
    print(f"  Input dim  : {actual_dim}")
    for i, (h, ctx) in enumerate(zip(HIDDEN_DIMS, TDNN_CONTEXTS)):
        print(f"  Layer {i+1}     : {h} unit  ctx={ctx}  LayerNorm=True")
    print(f"  Output     : {len(vocab)} unit (fonem+blank)  LayerNorm=False")
    print(f"  Dropout    : {DROPOUT_RATE}  |  Beam: {BEAM_WIDTH}")
    print(f"  FIX 4 ctx2 : [-2,0] bukan [-1,2] (history-focused)")

    model = TDNNModel(
        input_dim=actual_dim, hidden_dims=HIDDEN_DIMS,
        contexts=TDNN_CONTEXTS, output_dim=len(vocab),
        dropout=DROPOUT_RATE, lr=LR_WARMUP,
    )

    history = train(model, train_ids, val_ids, sequences,
                    targets, word_refs, id2phn)
    pd.DataFrame(history).to_csv(OUT_HISTORY_CSV, index=False)
    print(f"  History tersimpan: {OUT_HISTORY_CSV}")

    if os.path.exists(MODEL_PATH):
        model.load(MODEL_PATH)

    print("\n"+"="*62)
    print("  TAHAP 3 — Evaluasi WER")
    print("="*62)
    wr_val,  _     = evaluate(model, val_ids,  sequences,
                               word_refs, id2phn, "val")
    wr_test, s_t   = evaluate(model, test_ids, sequences,
                               word_refs, id2phn, "test")

    all_rows = wr_val + wr_test
    df_w     = pd.DataFrame(all_rows)
    df_w[["audio_id","WER","S","D","I","N"]].to_csv(OUT_WER_CSV, index=False)
    df_w[["audio_id","referensi","hipotesis","WER"]].to_csv(
        OUT_DEC_CSV, index=False)
    print(f"\n  {OUT_WER_CSV}\n  {OUT_DEC_CSV}")

    print("\n"+"="*62)
    print("  TAHAP 4 — Perbandingan GMM vs TDNN-CTC v6")
    print("="*62)
    print_comparison(s_t)

    return model, s_t, history


if __name__ == "__main__":
    model, summary, history = main()